# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghayoorahmed7/flyrank_ml_intership/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [7]:
# Confirm the classification framing: trend_direction is categorical with a clear
# binary split we care about (down vs not-down), not a continuous score or unlabeled data.
import pandas as pd
from pathlib import Path

CANDIDATES = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "content_refresh_anonymized.csv",
]
DATA_PATH = next((p for p in CANDIDATES if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Couldn't find content_refresh_anonymized.csv in any of: "
        + ", ".join(CANDIDATES)
        + " -- update CANDIDATES with the real path in your repo."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded from:", DATA_PATH)
print("trend_direction values:", df["trend_direction"].unique().tolist())

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print("\nClass balance for the classification target:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))


Loaded from: content_refresh_anonymized.csv
trend_direction values: ['down', 'stable', 'new', 'up', 'flat']

Class balance for the classification target:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [8]:
# Show where the proxy label comes from and how common it is.
print("Target column: is_declining_label")
print("Defined as: trend_direction == 'down' (a CURRENT-window bucket, not a future outcome)\n")

print(df["is_declining_label"].value_counts())
print(f"\n{df['is_declining_label'].mean()*100:.1f}% of pages are currently labeled declining.")
print("(Matches the 54.2% figure from w01_research_question.ipynb.)")


Target column: is_declining_label
Defined as: trend_direction == 'down' (a CURRENT-window bucket, not a future outcome)

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

54.2% of pages are currently labeled declining.
(Matches the 54.2% figure from w01_research_question.ipynb.)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [9]:
# A naive, rule-based "top 50" as a reference point, computed directly from this data --
# not a trained model yet, just a sanity check on how a single-signal rule would perform,
# to set up the comparison the metric is meant to support.
naive_top50_precision = []
for client_id, group in df.groupby("client_id"):
    naive_top50 = group.sort_values("impressions_90d", ascending=False).head(50)
    if len(naive_top50) > 0:
        precision = naive_top50["is_declining_label"].mean()
        naive_top50_precision.append(precision)

import numpy as np
print("Naive rule: 'top 50 pages by impressions_90d, per client'")
print(f"Median Precision@50 across clients: {np.median(naive_top50_precision):.3f}")
print("\nReference from the FlyRank guide's starter pipeline (30k-row starter slice, client-holdout):")
print("  fixed-rule baseline   Precision@50 = 0.240")
print("  random forest model   Precision@50 = 0.740")


Naive rule: 'top 50 pages by impressions_90d, per client'
Median Precision@50 across clients: 0.460

Reference from the FlyRank guide's starter pipeline (30k-row starter slice, client-holdout):
  fixed-rule baseline   Precision@50 = 0.240
  random forest model   Precision@50 = 0.740


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [10]:
# Show the unit of analysis as an actual dataframe.
print("Shape:", df.shape)
print("Unique (client_id, content_id) pairs:", df.drop_duplicates(subset=["client_id", "content_id"]).shape[0])
print("Duplicate page rows:", df.duplicated(subset=["client_id", "content_id"]).sum())

cols_to_show = [c for c in [
    "client_id", "content_id", "trend_direction", "is_declining_label",
    "impressions_90d", "sessions_90d", "content_age_days", "impression_tier"
] if c in df.columns]

df[cols_to_show].head(10)


Shape: (30000, 45)
Unique (client_id, content_id) pairs: 30000
Duplicate page rows: 0


,client_id,content_id,trend_direction,is_declining_label,impressions_90d,sessions_90d,content_age_days,impression_tier
0,client_f369cb89fc,content_304f48230142,down,1,3803,17,187,good
1,client_4e07408562,content_a1fb4e703a9e,down,1,15320,9,445,good
2,client_7f2253d7e2,content_9aa793d4d895,down,1,12581,11,141,good
3,client_19581e27de,content_331d6c4de07b,stable,0,11751,78,463,good
4,client_3fdba35f04,content_d99b7a2d90ca,down,1,19140,145,263,good
5,client_f369cb89fc,content_d4084a4bc775,down,1,3970,5,147,good
6,client_8722616204,content_9a34b442b552,down,1,20,1,90,low
7,client_19581e27de,content_a63219c6e95a,stable,0,1724,28,445,moderate
8,client_6208ef0f77,content_5e6c160719bc,down,1,32574,68,90,excellent
9,client_19581e27de,content_c27558df2b0c,down,1,1240,3,257,moderate


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [11]:
# Show that no single signal cleanly separates decliners from non-decliners --
# the justification for why a fixed rule underperforms a learned combination.
numeric_candidates = [c for c in ["impressions_90d", "sessions_90d", "content_age_days"] if c in df.columns]

for col in numeric_candidates:
    declining_mean = df.loc[df["is_declining_label"] == 1, col].mean()
    stable_mean = df.loc[df["is_declining_label"] == 0, col].mean()
    print(f"{col}: mean for declining pages = {declining_mean:.1f} | mean for non-declining = {stable_mean:.1f}")

print("\nIf a single signal separated the classes cleanly, these means would be far apart.")
print("They aren't -- which is why a one-line rule underperforms a learned, weighted combination.")


impressions_90d: mean for declining pages = 4919.1 | mean for non-declining = 5533.3
sessions_90d: mean for declining pages = 34.8 | mean for non-declining = 39.8
content_age_days: mean for declining pages = 236.2 | mean for non-declining = 279.8

If a single signal separated the classes cleanly, these means would be far apart.
They aren't -- which is why a one-line rule underperforms a learned, weighted combination.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.